# 02_EDA_Analysis

Rossmann Sales Forecasting — exploratory analysis and business-oriented feature/target review. This notebook uses the completed cleaning logic and preserves the project findings without inventing additional analyses.


## EDA Setup


In [ ]:
import pandas as pd

train = pd.read_csv('/content/train.csv')
store = pd.read_csv('/content/store.csv')

print("Train dataset shape:", train.shape)
print("Store dataset shape:", store.shape)

Train dataset shape: (1017209, 9)
Store dataset shape: (1115, 10)


/tmp/ipykernel_4891/1444318771.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train.csv')


In [ ]:
train['Date'] = pd.to_datetime(train['Date'])

print(train['Date'].dtype)
print(train['Date'].min())
print(train['Date'].max())

datetime64[ns]
2013-01-01 00:00:00
2015-07-31 00:00:00


In [ ]:
train['Year'] = train['Date'].dt.year

print(train[['Date', 'Year']].head())

        Date  Year
0 2015-07-31  2015
1 2015-07-31  2015
2 2015-07-31  2015
3 2015-07-31  2015
4 2015-07-31  2015


In [ ]:
train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day
train['WeekOfYear'] = train['Date'].dt.isocalendar().week.astype(int)

print(train[['Date', 'Year', 'Month', 'Day', 'WeekOfYear']].head())

        Date  Year  Month  Day  WeekOfYear
0 2015-07-31  2015      7   31          31
1 2015-07-31  2015      7   31          31
2 2015-07-31  2015      7   31          31
3 2015-07-31  2015      7   31          31
4 2015-07-31  2015      7   31          31


In [ ]:
train['IsWeekend'] = train['DayOfWeek'].isin([6, 7]).astype(int)

print(train[['DayOfWeek', 'IsWeekend']].drop_duplicates().sort_values('DayOfWeek'))

      DayOfWeek  IsWeekend
4460          1          0
3345          2          0
2230          3          0
1115          4          0
0             5          0
6690          6          1
5575          7          1


In [ ]:
train['YearMonth'] = train['Date'].dt.to_period('M').astype(str)

print(train[['Date', 'YearMonth']].head())

        Date YearMonth
0 2015-07-31   2015-07
1 2015-07-31   2015-07
2 2015-07-31   2015-07
3 2015-07-31   2015-07
4 2015-07-31   2015-07


In [ ]:
merged = train.merge(
    store,
    on='Store',
    how='left'
)

print("Merged dataset shape:", merged.shape)
print("\nMerged columns:")
print(merged.columns.tolist())

Merged dataset shape: (1017209, 24)

Merged columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'YearMonth', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


In [ ]:
merged['StateHoliday'] = merged['StateHoliday'].astype(str)

print(merged['StateHoliday'].map(type).value_counts())
print("\nStateHoliday values:")
print(merged['StateHoliday'].value_counts())

StateHoliday
<class 'str'>    1017209
Name: count, dtype: int64

StateHoliday values:
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64


In [ ]:
# Create a working copy for machine learning
df = merged.copy()

print("Final working dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

Final working dataset shape: (1017209, 24)
Total missing values: 2173431


## Sales Distribution & Business Checks


In [ ]:
print("Sales statistics:")
print(train['Sales'].describe())

print("\nNumber of zero-sales records:")
print((train['Sales'] == 0).sum())

print("\nNumber of negative-sales records:")
print((train['Sales'] < 0).sum())

Sales statistics:
count    1.017209e+06
mean     5.773819e+03
std      3.849926e+03
min      0.000000e+00
25%      3.727000e+03
50%      5.744000e+03
75%      7.856000e+03
max      4.155100e+04
Name: Sales, dtype: float64

Number of zero-sales records:
172871

Number of negative-sales records:
0


In [ ]:
print("Sales = 0 when store is closed:")
print(train.loc[train['Open'] == 0, 'Sales'].eq(0).all())

print("\nNumber of closed-store records:")
print((train['Open'] == 0).sum())

print("\nNumber of closed-store records with non-zero sales:")
print(((train['Open'] == 0) & (train['Sales'] != 0)).sum())

Sales = 0 when store is closed:
True

Number of closed-store records:
172817

Number of closed-store records with non-zero sales:
0


In [ ]:
open_data = merged[merged['Open'] == 1]

print("Open-store records:", len(open_data))
print("Zero-sales records while open:", (open_data['Sales'] == 0).sum())
print("Average sales while open:", open_data['Sales'].mean())
print("Maximum sales while open:", open_data['Sales'].max())

Open-store records: 844392
Zero-sales records while open: 54
Average sales while open: 6955.514290755952
Maximum sales while open: 41551


## Categorical Feature Review


In [ ]:
print("StoreType:")
print(merged['StoreType'].value_counts())

print("\nAssortment:")
print(merged['Assortment'].value_counts())

print("\nStateHoliday:")
print(merged['StateHoliday'].value_counts())

print("\nPromoInterval:")
print(merged['PromoInterval'].value_counts(dropna=False))

StoreType:
StoreType
a    551627
d    312912
c    136840
b     15830
Name: count, dtype: int64

Assortment:
Assortment
a    537445
c    471470
b      8294
Name: count, dtype: int64

StateHoliday:
StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64

PromoInterval:
PromoInterval
NaN                 508031
Jan,Apr,Jul,Oct     293122
Feb,May,Aug,Nov     118596
Mar,Jun,Sept,Dec     97460
Name: count, dtype: int64


## Target & Correlation Analysis


In [ ]:
target = df['Sales']

print("Target name:", target.name)
print("Target shape:", target.shape)
print("Target data type:", target.dtype)

Target name: Sales
Target shape: (1017209,)
Target data type: int64


In [ ]:
correlation = df[['Sales', 'Customers']].corr()

print(correlation)

              Sales  Customers
Sales      1.000000   0.894711
Customers  0.894711   1.000000


## Time Coverage & Year Review


In [ ]:
print("Minimum date:", df['Date'].min())
print("Maximum date:", df['Date'].max())

print("\nNumber of unique dates:", df['Date'].nunique())

print("\nRecords by year:")
print(df['Year'].value_counts().sort_index())

Minimum date: 2013-01-01 00:00:00
Maximum date: 2015-07-31 00:00:00

Number of unique dates: 942

Records by year:
Year
2013    406974
2014    373855
2015    236380
Name: count, dtype: int64


## EDA Findings & Modelling Implications

The exploratory checks establish several important business and modelling observations:

- The training period runs from **2013-01-01 to 2015-07-31** with 942 unique dates.
- Sales contains substantial zero values, driven primarily by closed stores.
- The `Open` indicator is therefore an important predictor and must be interpreted carefully: its high importance later reflects the strong distinction between open and closed stores, not a causal effect estimate.
- `Sales` and `Customers` have a strong positive relationship (correlation ≈ **0.895**), confirming that customer volume is closely associated with sales. `Customers` is not used as an input to the initial Sales model because future customer counts are not available at prediction time.
- Store type, assortment, promotions, holidays, competition and calendar variables provide useful business context for forecasting.

### Modelling hand-off

The EDA supports a tree-based supervised model for store-day Sales prediction and a separate time-series approach for aggregate daily Sales. The ML notebook uses a chronological split to avoid future-to-past leakage.

## Optional EDA Visualizations

The following simple charts can be executed when the source CSV files are available. They are intentionally beginner-friendly and avoid violin plots/heatmaps. They provide univariate, bivariate and time-trend views for presentation purposes.

In [ ]:
import matplotlib.pyplot as plt

# Monthly Sales trend
monthly_sales = (
    merged.groupby('YearMonth', as_index=False)['Sales']
    .sum()
    .sort_values('YearMonth')
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['YearMonth'], monthly_sales['Sales'])
plt.title('Monthly Total Sales Trend')
plt.xlabel('Year-Month')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Sales by Store Type
store_type_sales = (
    merged.groupby('StoreType', as_index=False)['Sales']
    .mean()
)

plt.figure(figsize=(8, 5))
plt.bar(store_type_sales['StoreType'], store_type_sales['Sales'])
plt.title('Average Sales by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Average Sales')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Sales vs Customers
plt.figure(figsize=(8, 5))
plt.scatter(df['Customers'], df['Sales'], alpha=0.2)
plt.title('Sales vs Customers')
plt.xlabel('Customers')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()


### Visualization note

These visualization cells are supplementary and do not alter the cleaned dataset or any model results. The project's Power BI dashboard provides the primary interactive visualization layer.

# EDA — Business-Focused Analysis

## Objective
Explore sales behaviour across time, promotions, holidays, customers, store characteristics, and competition-related variables before model development.

## Required business questions
1. How does sales behaviour change over time?
2. How do promotions relate to sales?
3. How do store types and assortments differ?
4. How do school and state holidays relate to sales?
5. How strongly are Customers and Sales related?
6. Does competition distance provide useful predictive information?

## Interpretation rule
The EDA is descriptive. Relationships observed here should not automatically be interpreted as causal effects. Model-based feature importance in Notebook 3 is used separately to understand which variables the trained model relies on.


In [ ]:
# Business-focused EDA visuals
import matplotlib.pyplot as plt

eda = df.copy()
eda["Date"] = pd.to_datetime(eda["Date"])

# 1. Annual Sales
annual_sales = eda.groupby(eda["Date"].dt.year)["Sales"].sum()
annual_sales.plot(kind="bar", title="Total Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

# 2. Monthly Sales
monthly_sales = eda.groupby(eda["Date"].dt.month)["Sales"].sum()
monthly_sales.plot(kind="bar", title="Total Sales by Month")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

# 3. Sales by Store Type
store_type_sales = eda.groupby("StoreType")["Sales"].mean().sort_values(ascending=False)
store_type_sales.plot(kind="bar", title="Average Daily Sales by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

# 4. Promotion comparison
promo_sales = eda.groupby("Promo")["Sales"].mean()
promo_sales.plot(kind="bar", title="Average Sales by Promotion Status")
plt.xlabel("Promo")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

# 5. School Holiday comparison
school_holiday_sales = eda.groupby("SchoolHoliday")["Sales"].mean()
school_holiday_sales.plot(kind="bar", title="Average Sales by School Holiday Status")
plt.xlabel("School Holiday")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

# 6. State Holiday comparison
state_holiday_sales = eda.groupby("StateHoliday")["Sales"].mean()
state_holiday_sales.plot(kind="bar", title="Average Sales by State Holiday")
plt.xlabel("State Holiday")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

# 7. Customers vs Sales
plt.figure()
plt.scatter(eda["Customers"], eda["Sales"], alpha=0.2)
plt.title("Sales vs Customers")
plt.xlabel("Customers")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# 8. Competition Distance vs Sales
open_eda = eda[eda["Open"] == 1].copy()
plt.figure()
plt.scatter(open_eda["CompetitionDistance"], open_eda["Sales"], alpha=0.2)
plt.title("Sales vs Competition Distance")
plt.xlabel("Competition Distance")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()
